# LeakageSafeGenerator Playground

Edit the sample and generation settings below, then run the notebook cells top to bottom. This notebook is intentionally small so you can poke the generation loop directly.

In [ ]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "method").exists():
    project_root = project_root.parent
if not (project_root / "method").exists():
    project_root = Path(r"H:/Project/traffic-hope-network")

sys.path.insert(0, str(project_root))
project_root

In [ ]:
from method.generate.modeling import load_model
from method.generate_new import (
    AttentionLeakageDetector,
    DecodeSettings,
    LeakageSafeGenerator,
    RepairSettings,
)

# Edit these paths/model keys for your run.
MODEL_PATH = "Qwen/Qwen3-4B"
TOKENIZER_PATH = None
MODEL_KEY = "Qwen3-4B"
DETECTOR_CONFIG = project_root / "method/calibrate_leakage_detector/output/detector_config.json"

DEVICE = "auto"
DTYPE = "bfloat16"  # auto, float32, float16, bfloat16
TRUST_REMOTE_CODE = True
LOCAL_FILES_ONLY = False

## Editable Toy Sample

Change `question` and `answer` directly here. The privileged prompt is formed as `clean_prompt + privileged_context`.

In [ ]:
sample = {
    "question": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
    "answer": "72",
}

clean_prompt = sample["question"].strip() + "\n\nExplain your solution step by step."
privileged_context = f" Given the ground truth answer is {sample['answer']}"
privileged_prompt = clean_prompt + privileged_context

print("Clean prompt:\n", clean_prompt)
print("\nPrivileged context:\n", privileged_context)
print("\nPrivileged prompt:\n", privileged_prompt)

In [ ]:
# Load model/tokenizer and detector. This can take a while for large models.
model, tokenizer = load_model(
    MODEL_PATH,
    tokenizer_path=TOKENIZER_PATH,
    device=DEVICE,
    dtype=DTYPE,
    trust_remote_code=TRUST_REMOTE_CODE,
    local_files_only=LOCAL_FILES_ONLY,
)
detector = AttentionLeakageDetector.from_file(DETECTOR_CONFIG, MODEL_KEY)

len(detector.heads), detector.threshold, detector.window_size, detector.aggregation

In [ ]:
# Keep this small while debugging the generation loop.
decode = DecodeSettings(
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    top_k=0,
    do_sample=True,
    enable_thinking=True,
    seed=42,
)

generator = LeakageSafeGenerator(
    model=model,
    tokenizer=tokenizer,
    detector=detector,
    decode=decode,
    repair=RepairSettings(),
)

In [ ]:
result = generator.generate(
    clean_prompt=clean_prompt,
    privileged_prompt=privileged_prompt,
    privileged_context=privileged_context,
    include_unfixed=True,
    show_progress=True,
)

print(result["text"])

In [ ]:
print("fixed token count:", len(result["token_ids"]))
print("repair events:", len(result["repair_events"]))

for i, event in enumerate(result["repair_events"], 1):
    print("=" * 80)
    print("event", i)
    print("detected_index:", event["detected_index"])
    print("start_index:", event["start_index"])
    print("detector_score:", event["detector_score"])
    print("removed:", event["removed_token_text"])
    print("replacement:", event.get("replacement_token_text", ""))
    print("raw event:")
    print(json.dumps(event, ensure_ascii=False, indent=2))

In [ ]:
# Optional: compare with the unrepaired privileged generation.
if result["unfixed"] is not None:
    print(result["unfixed"]["text"])